# E3 — Multi-horizon supervised datasets (15 / 30 / 60 / 120 minutes)

**Companion to Notebook 06 — does not overwrite existing 15-minute outputs.**

- Reads the same cleaned complete-case input as Notebook 06.
- Uses the **original Notebook 06 chronological cutoff dates** when `outputs/06_common_experiment/06_split_metadata.json` exists. If absent, reconstructs cutoffs using the original 15-minute valid input timestamps (same logic as Notebook 06).
- Creates targets strictly within the same UUID and contiguous segment, requiring an **exact elapsed interval** and uninterrupted 15-minute observations throughout the forecast horizon.
- Requires input and target timestamps to belong to the same chronological partition.
- Saves separate files per horizon under `outputs/E3_multihorizon/hXX/`.
- Does not train models. Notebooks 07A/07B/07C and 08A/08B/8C need horizon-aware adaptations before performance claims can be made.

**Important:** The full 5-step history check remains the responsibility of downstream representation notebooks. The same per-horizon target-validity rules are applied to all representations.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
INPUT_FILE = PROJECT_DIR / "outputs" / "05_ml_dataset" / "06_ai_ready_complete_case.csv"
BASE_METADATA = PROJECT_DIR / "outputs" / "06_common_experiment" / "06_split_metadata.json"
OUT_ROOT = PROJECT_DIR / "outputs" / "E3_multihorizon"
HORIZONS = (15, 30, 60, 120)
STEP = pd.Timedelta(minutes=15)
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15
assert INPUT_FILE.exists(), f"Run Notebook 05 first: {INPUT_FILE}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
required = {"uuid", "timestamp", "segment_id", "temperature", "relative_humidity", "co2"}
assert not (required - set(df.columns)), f"Missing: {required - set(df.columns)}"
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
for col in ("temperature", "relative_humidity", "co2"):
    df[col] = pd.to_numeric(df[col], errors="coerce")
assert df[["uuid", "segment_id", "timestamp", "temperature", "relative_humidity", "co2"]].notna().all().all()
assert not df.duplicated(["uuid", "timestamp"]).any(), "Duplicate device/timestamp pairs"
df = df.sort_values(["uuid", "segment_id", "timestamp"]).reset_index(drop=True)

# Reproduce Notebook 06's original 15-min target-eligible timestamp universe.
g = df.groupby(["uuid", "segment_id"], sort=False)
next_ts = g["timestamp"].shift(-1)
base_eligible = (next_ts == df["timestamp"] + STEP)
base_times = np.array(sorted(df.loc[base_eligible, "timestamp"].unique()))
assert len(base_times) >= 3
if BASE_METADATA.exists():
    meta = json.loads(BASE_METADATA.read_text(encoding="utf-8"))
    train_end = pd.Timestamp(meta["train_end"])
    val_end = pd.Timestamp(meta["validation_end"])
    cutoff_source = "original Notebook 06 metadata"
else:
    train_end = pd.Timestamp(base_times[max(0, int(np.floor(len(base_times) * TRAIN_RATIO)) - 1)])
    vi = min(max(int(np.floor(len(base_times) * (TRAIN_RATIO + VAL_RATIO))) - 1, 1), len(base_times) - 2)
    val_end = pd.Timestamp(base_times[vi])
    cutoff_source = "reconstructed from Notebook 06's 15-min eligible timestamps"
assert train_end < val_end
print("Cutoffs:", train_end, val_end, "|", cutoff_source)
print("Input rows:", len(df), "| devices:", df.uuid.nunique())

In [ ]:
def make_horizon(horizon_minutes):
    assert horizon_minutes > 0 and horizon_minutes % 15 == 0
    steps = horizon_minutes // 15
    group = df.groupby(["uuid", "segment_id"], sort=False)
    target_ts = group["timestamp"].shift(-steps)
    target_temp = group["temperature"].shift(-steps)

    # Every intermediate step must be exactly 15 min (not merely the endpoint).
    contiguous = pd.Series(True, index=df.index)
    for k in range(1, steps + 1):
        t_k = group["timestamp"].shift(-k)
        contiguous &= t_k.eq(df["timestamp"] + k * STEP)

    valid = contiguous & target_ts.notna() & target_temp.notna()
    d = df.loc[valid, ["uuid", "timestamp", "segment_id", "temperature", "relative_humidity", "co2"]].copy()
    d["target_timestamp"] = target_ts.loc[valid]
    target_col = f"target_temperature_t_plus_{horizon_minutes}"
    d[target_col] = target_temp.loc[valid]
    d["forecast_horizon_minutes"] = horizon_minutes
    d["split"] = np.select(
        [d["timestamp"] <= train_end, d["timestamp"] <= val_end],
        ["train", "validation"], default="test"
    )
    # Do not allow the target to cross a partition boundary.
    inside = (
        ((d["split"] == "train") & (d["target_timestamp"] <= train_end)) |
        ((d["split"] == "validation") & (d["target_timestamp"] > train_end) & (d["target_timestamp"] <= val_end)) |
        ((d["split"] == "test") & (d["target_timestamp"] > val_end))
    )
    boundary_excluded = int((~inside).sum())
    d = d.loc[inside].sort_values(["timestamp", "uuid"]).reset_index(drop=True)
    assert not d.duplicated(["uuid", "timestamp"]).any()
    assert ((d["target_timestamp"] - d["timestamp"]) == pd.Timedelta(minutes=horizon_minutes)).all()
    assert d[target_col].notna().all()
    assert d["split"].isin(["train", "validation", "test"]).all()
    return d, target_col, boundary_excluded

summary = []
for horizon in HORIZONS:
    d, target_col, boundary_excluded = make_horizon(horizon)
    out = OUT_ROOT / f"h{horizon:03d}"
    out.mkdir(parents=True, exist_ok=True)
    d.to_csv(out / "01_common_supervised_dataset.csv", index=False)
    for name in ("train", "validation", "test"):
        d.loc[d.split.eq(name)].to_csv(out / f"03_{name}_common.csv", index=False)
    audit = (d.groupby("split", observed=True)
               .agg(samples=("uuid", "size"), devices=("uuid", "nunique"),
                    input_start=("timestamp", "min"), input_end=("timestamp", "max"),
                    target_start=("target_timestamp", "min"), target_end=("target_timestamp", "max"))
               .reset_index())
    audit.to_csv(out / "04_chronological_split_audit.csv", index=False)
    metadata = {"horizon_minutes": horizon, "target_column": target_col,
                "train_end": str(train_end), "validation_end": str(val_end),
                "cutoff_source": cutoff_source, "boundary_samples_removed": boundary_excluded,
                "n_samples": len(d), "n_devices": int(d.uuid.nunique()),
                "split_counts": {k: int((d.split == k).sum()) for k in ("train", "validation", "test")},
                "source_file": str(INPUT_FILE), "scaling_applied": False,
                "full_history_check": "downstream representation construction required"}
    (out / "06_split_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    summary.append({"horizon_min": horizon, "samples": len(d), "boundary_removed": boundary_excluded, **metadata["split_counts"]})

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_ROOT / "00_multihorizon_summary.csv", index=False)
print(summary_df.to_string(index=False))
print("Outputs:", OUT_ROOT)

## Controls before training

1. **No comparison of raw RMSE across horizons without the corresponding persistence baseline.** Compute RMSE skill = `1 - RMSE_model / RMSE_persistence` on exactly matched samples.
2. **Equal test keys per pipeline:** for each horizon, intersect and assert identical `(uuid, timestamp, target_timestamp)` before computing metrics; the graph pipeline may have fewer common timestamps.
3. **History leakage:** all five input observations must be within the relevant partition; apply the same history eligibility to all representations.
4. **Sensor quantization:** estimate observed measurement increments and report resolution only if supported by device metadata; empirical increments are not proof of hardware resolution.
5. **No results are claimed by this notebook:** it only prepares targets and split audits.
